In [2]:
import os
import numpy as np
import pandas as pd
from pathlib import Path
from astropy.io import fits

# ============================================================
# SHARED SETUP — object-centered tiling for debvader
# (debvader was trained on cutouts centered on the target galaxy —
#  a fixed grid tile is NOT compatible with its spatial assumptions)
# ============================================================
tract_id = "3828"
bands = ['u', 'g', 'r', 'i', 'z', 'y']
tile_size = 59
half_tile = tile_size // 2   # 29 — center pixel offset
patch_spacing = 4000

patch_tiers = {
    "HIGH": ['0,6', '0,0', '1,0', '3,4', '0,2'],
    "MEAN": ['5,0', '5,1', '6,0', '4,1', '4,3'],
    "LOW":  ['2,4', '1,3', '3,1', '6,1', '4,6'],
}

catalog = pd.read_parquet(r"processed/cleaned_catalog_with_blendedness_truth.parquet")


def process_patch_centered(patch_id, quality_tier):
    raw_images_dir = os.path.join(r"RAW", r"IMAGES", quality_tier)
    tiles_out_dir = Path("processed") / "TILES_TF" / quality_tier / patch_id
    tiles_out_dir.mkdir(parents=True, exist_ok=True)

    print(f"=== PATCH {patch_id} ({quality_tier}) ===")

    patch_cat = catalog[catalog['patch_true'] == patch_id].copy()
    print(f"Found {len(patch_cat)} total objects in patch {patch_id}.")

    x_raw = patch_cat['x'].values
    y_raw = patch_cat['y'].values
    obj_ids = patch_cat['id'].values

    patch_col, patch_row = map(int, patch_id.split(','))
    x_offset = patch_col * patch_spacing
    y_offset = patch_row * patch_spacing
    x_coords = x_raw - x_offset
    y_coords = y_raw - y_offset

    print(f"Loading FITS images for patch {patch_id} ({quality_tier})...")
    image_data = {}
    img_h, img_w = None, None
    for band in bands:
        file_name = f"calexp-{band}-{tract_id}-{patch_id}.fits"
        fits_path = os.path.join(raw_images_dir, band.upper(), file_name)
        with fits.open(fits_path) as hdul:
            image_data[band] = hdul[1].data
            img_h, img_w = image_data[band].shape

    saved_tiles = 0
    skipped_edge = 0

    for i in range(len(patch_cat)):
        cx, cy = x_coords[i], y_coords[i]
        obj_id = obj_ids[i]

        # integer center pixel, cutout box centered on it
        cx_int, cy_int = int(round(cx)), int(round(cy))
        x_min, x_max = cx_int - half_tile, cx_int - half_tile + tile_size
        y_min, y_max = cy_int - half_tile, cy_int - half_tile + tile_size

        # skip objects too close to the patch edge for a full 59x59 cutout
        if x_min < 0 or y_min < 0 or x_max > img_w or y_max > img_h:
            skipped_edge += 1
            continue

        tile_bands = [image_data[band][y_min:y_max, x_min:x_max] for band in bands]
        stacked_cf = np.stack(tile_bands, axis=0)
        stacked_cf = np.nan_to_num(stacked_cf, nan=0.0, posinf=0.0, neginf=0.0)
        stacked_cl = np.transpose(stacked_cf, (1, 2, 0)).astype(np.float32)  # (59, 59, 6)

        # filename keeps the object id AND its true local center pixel,
        # so forced photometry can locate it at exactly (29, 29) in every tile
        np.save(tiles_out_dir / f"tile_obj{obj_id}_cx{cx_int}_cy{cy_int}.npy", stacked_cl)
        saved_tiles += 1

    print(f"Saved: {saved_tiles} | Skipped (edge): {skipped_edge} | -> {tiles_out_dir}\n")
    return saved_tiles, skipped_edge


# ============================================================
# RUN ALL 15 PATCHES
# ============================================================
results = {}
for tier, patch_list in patch_tiers.items():
    for patch_id in patch_list:
        saved, skipped = process_patch_centered(patch_id, tier)
        results[(tier, patch_id)] = (saved, skipped)

print("="*60)
print("SUMMARY")
print("="*60)
total_saved = sum(s for s, d in results.values())
total_skipped = sum(d for s, d in results.values())
for (tier, patch_id), (saved, skipped) in results.items():
    print(f"{tier:5s} {patch_id:6s} saved={saved:5d} skipped_edge={skipped:5d}")
print(f"\nTOTAL saved: {total_saved:,} | TOTAL skipped (edge): {total_skipped:,}")

=== PATCH 0,6 (HIGH) ===
Found 2026 total objects in patch 0,6.
Loading FITS images for patch 0,6 (HIGH)...
Saved: 2002 | Skipped (edge): 24 | -> processed\TILES_TF\HIGH\0,6

=== PATCH 0,0 (HIGH) ===
Found 2224 total objects in patch 0,0.
Loading FITS images for patch 0,0 (HIGH)...
Saved: 2224 | Skipped (edge): 0 | -> processed\TILES_TF\HIGH\0,0

=== PATCH 1,0 (HIGH) ===
Found 2703 total objects in patch 1,0.
Loading FITS images for patch 1,0 (HIGH)...
Saved: 2674 | Skipped (edge): 29 | -> processed\TILES_TF\HIGH\1,0

=== PATCH 3,4 (HIGH) ===
Found 2954 total objects in patch 3,4.
Loading FITS images for patch 3,4 (HIGH)...
Saved: 2907 | Skipped (edge): 47 | -> processed\TILES_TF\HIGH\3,4

=== PATCH 0,2 (HIGH) ===
Found 2655 total objects in patch 0,2.
Loading FITS images for patch 0,2 (HIGH)...
Saved: 2633 | Skipped (edge): 22 | -> processed\TILES_TF\HIGH\0,2

=== PATCH 5,0 (MEAN) ===
Found 2487 total objects in patch 5,0.
Loading FITS images for patch 5,0 (MEAN)...
Saved: 2471 | Skip

In [3]:
import pandas as pd
from pathlib import Path
import re

pattern = re.compile(r"^tile_obj(\d+)_cx(-?\d+)_cy(-?\d+)$")

def build_manifest_centered(tiles_root):
    records = []
    for tier_dir in tiles_root.iterdir():
        if not tier_dir.is_dir():
            continue
        tier = tier_dir.name
        for patch_dir in tier_dir.iterdir():
            if not patch_dir.is_dir():
                continue
            patch_id = patch_dir.name
            for f in patch_dir.glob("*.npy"):
                m = pattern.match(f.stem)
                if not m:
                    print(f"Unrecognized filename, skipping: {f}")
                    continue
                obj_id, cx, cy = int(m.group(1)), int(m.group(2)), int(m.group(3))
                records.append({
                    "tile_path": str(f),
                    "tier": tier,
                    "patch_id": patch_id,
                    "object_id": obj_id,
                    "center_x": cx,
                    "center_y": cy,
                })
    return pd.DataFrame(records)


tiles_tf_root = Path(r"processed/TILES_TF")
manifest_tf = build_manifest_centered(tiles_tf_root)

print("=== TILES_TF manifest (object-centered) ===")
print(f"Total tiles: {len(manifest_tf):,}")
print(manifest_tf["tier"].value_counts())
print(manifest_tf.groupby("patch_id").size().sort_values(ascending=False))

# reapply your existing patch-level split assignment — no need to re-split,
# since splitting was done at the patch level and patches haven't changed
split_assignment = pd.read_parquet(r"processed/patch_split_assignment.parquet")
split_map = dict(zip(split_assignment["patch_id"], split_assignment["split"]))
manifest_tf["split"] = manifest_tf["patch_id"].map(split_map)

print("\nSplit tile counts:")
print(manifest_tf.groupby("split").size())

manifest_tf.to_parquet(r"processed/tile_manifest_tf.parquet", index=False)
print("\nSaved: processed/tile_manifest_tf.parquet")

=== TILES_TF manifest (object-centered) ===
Total tiles: 38,321
tier
MEAN    13188
LOW     12693
HIGH    12440
Name: count, dtype: int64
patch_id
2,4    2960
4,3    2931
4,1    2915
3,4    2907
3,1    2800
1,3    2766
5,1    2714
1,0    2674
0,2    2633
5,0    2471
0,0    2224
6,0    2157
4,6    2127
6,1    2040
0,6    2002
dtype: int64

Split tile counts:
split
test      6843
train    22954
val       8524
dtype: int64

Saved: processed/tile_manifest_tf.parquet
